# Joins and Windows

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

spark = SparkSession.builder.appName("module-03-dataframes").master("local[*]").getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "4")

In [ ]:
municipalities_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("municipality_name", StringType(), True),
    StructField("canton", StringType(), True),
    StructField("population", IntegerType(), True),
])
accessibility_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("accessibility_score", DoubleType(), True),
])
poi_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("poi_count", IntegerType(), True),
])
property_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("property_value_index", DoubleType(), True),
])
transactions_schema = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("municipality_id", IntegerType(), True),
    StructField("property_id", IntegerType(), True),
    StructField("sale_price", IntegerType(), True),
    StructField("sale_date", StringType(), True),
    StructField("property_type", StringType(), True),
])
pop_history_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("year", IntegerType(), True),
    StructField("population", IntegerType(), True),
])

municipalities = spark.read.option("header", True).schema(municipalities_schema).csv("datasets/module_02/municipalities.csv")
accessibility_scores = spark.read.option("header", True).schema(accessibility_schema).csv("datasets/module_02/accessibility_scores.csv")
poi_counts = spark.read.option("header", True).schema(poi_schema).csv("datasets/module_02/poi_counts.csv")
property_values = spark.read.option("header", True).schema(property_schema).csv("datasets/module_02/property_values.csv")
transactions = spark.read.option("header", True).schema(transactions_schema).csv("datasets/module_03/transactions.csv").withColumn("sale_date", F.to_date("sale_date"))
population_history = spark.read.option("header", True).schema(pop_history_schema).csv("datasets/module_03/population_history.csv")

for name, df in {
    "municipalities": municipalities,
    "accessibility_scores": accessibility_scores,
    "poi_counts": poi_counts,
    "property_values": property_values,
    "transactions": transactions,
    "population_history": population_history,
}.items():
    df.createOrReplaceTempView(name)

In [ ]:
profile = transactions.join(municipalities.select("municipality_id", "municipality_name", "canton"), "municipality_id", "inner")     .join(property_values, "municipality_id", "left")     .select("transaction_id", "municipality_id", "municipality_name", "canton", "sale_price", "sale_date", "property_type", "property_value_index")

print("transactions", transactions.count())
print("profile", profile.count())
profile.show(10)

w = Window.partitionBy("canton").orderBy(F.desc("sale_price"), F.asc("transaction_id"))
top3 = profile.withColumn("sale_rank_in_canton", F.row_number().over(w)).filter(F.col("sale_rank_in_canton") <= 3)
top3.orderBy("canton", "sale_rank_in_canton").show(50)
top3.explain("formatted")